# **Libraries**

In [ ]:
# ---------------- Standard Libraries ----------------
import os
import pickle
from pathlib import Path
from datetime import datetime
import time
# ---------------- Data Handling ----------------
import numpy as np
import pandas as pd

# ---------------- Visualization ----------------
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'font.size': 9,
    'font.weight': 'bold',
    'font.family': 'serif'
})

# ---------------- Machine Learning (sklearn) ----------------
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, average_precision_score
)
# ---------------- Deep Learning (PyTorch) ----------------
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import _LRScheduler
from torchsummary import summary


# ---------------- File Handling ----------------
import h5py
import hdf5storage

# ---------------- Utilities ----------------
from tqdm import tqdm
import joblib
import shap

# ---------------- Custom Function ----------------
def load_pickle(filepath):
    """
    Load a Python object from a pickle file.

    Parameters
    ----------
    filepath : str
        Path to the .pkl file.

    Returns
    -------
    obj : any
        The Python object stored in the pickle file.
    """
    with open(filepath, 'rb') as f:
        obj = pickle.load(f)
    return obj

def extract_pid(filename):
    part = filename.split('_')[0]          
    pid_full = part.split('-')[1]          
    pid = pid_full[5:]                     
    return pid


# **Load the Full Dataset**
**M12**
- Good neurological state (0): RASS (0), GCS (13, 14, 15), CAMS (0, 1), ICANS (0, 1)
- Moderate state (1): RASS (-3, -2, -1), GCS (9, 10, 11, 12), CAMS (2, 3, 4), ICANS (2)
- Bad state (2): RASS (-5, -4), GCS (3, 4, 5, 6, 7, 8), CAMS (5, 6, 7), ICANS (3, 4)

In [ ]:
def group_gcs(y):
    if 3 <= y <= 8: return 2
    elif 9 <= y <= 12: return 1
    elif 13 <= y <= 15: return 0
    return None

def group_rass(y):
    if -5 <= y <= -4: return 2
    elif -3 <= y <= -1: return 1
    elif y == 0: return 0
    return None

def group_icans(y):
    if 0 <= y <= 1: return 0
    elif y == 2: return 1
    elif 3 <= y <= 4: return 2
    return None

def group_cams(x):
    if 0 <= x <= 1: return 0
    elif 2 <= x <= 4: return 1
    elif 5 <= x <= 7: return 2
    return None

In [ ]:
# ------------------- CAMS - Dataset --------------------------
df_CAMS_metadata = pd.read_csv('/home/ayush/Desktop/CAM-S_dataset/CAMS_Training_Final_Metadata.csv')
df_CAMS_metadata["GroupedScore"] = df_CAMS_metadata["CAMS_SF"].apply(group_cams)
df_CAMS_metadata = df_CAMS_metadata.rename(columns={'CAMS_SF': 'RawScore'})
df_CAMS_metadata['Dataset'] = 'CAMS'
df_CAMS_metadata['TransformedScore'] = df_CAMS_metadata['RawScore']
df_CAMS_metadata = df_CAMS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

# ------------------- ICANS - Dataset --------------------------
df_ICANS_metadata = pd.read_csv('/home/ayush/Desktop/ICANS_dataset/ICANSTraining_Final_Metadata.csv')
df_ICANS_metadata["GroupedScore"] = df_ICANS_metadata["ICANS_raw"].apply(group_icans)
df_ICANS_metadata = df_ICANS_metadata.rename(columns={'StudyID':'BDSPPatientID',                                                      
                                                      'ICANS_raw': 'RawScore'})
df_ICANS_metadata['Dataset'] = 'ICANS'
df_ICANS_metadata['TransformedScore'] = df_ICANS_metadata['RawScore']
df_ICANS_metadata = df_ICANS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

# ------------------- GCS - Dataset --------------------------
df_GCS_metadata = pd.read_csv('/home/ayush/Desktop/MGB_GCS_EEGs/GCS_Filename_with_QC2.csv')
df_GCS_metadata['Filename'] = df_GCS_metadata['Filename'].astype(str) + '.csv'
df_GCS_metadata['GroupedScore'] = df_GCS_metadata['GCS_value'].apply(group_gcs)
df_GCS_metadata['BDSPPatientID'] = df_GCS_metadata['Filename'].apply(extract_pid)
df_GCS_metadata = df_GCS_metadata.rename(columns={'Filename': 'MorgothOutputFilename',
                                                  'GCS_value': 'RawScore'})
df_GCS_metadata['Dataset'] = 'GCS'
df_GCS_metadata['TransformedScore'] = 15-df_GCS_metadata['RawScore']
df_GCS_metadata = df_GCS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

# ------------------- RASS - Dataset --------------------------
df_RASS_metadata=pd.read_csv('/home/ayush/Desktop/MGB_RASS_EEGs/RASS_ALL_EEG_10min/RASS_files_updated_all.csv')
df_RASS_metadata['Filename'] = df_RASS_metadata['Filename'].str.rstrip("'") + '.csv'
df_RASS_metadata = df_RASS_metadata[
    ~df_RASS_metadata['RASS_value'].isin([1, 2, 3, 4])
]
df_RASS_metadata['GroupedScore'] = df_RASS_metadata['RASS_value'].apply(group_rass)
df_RASS_metadata['BDSPPatientID'] = df_RASS_metadata['Filename'].apply(extract_pid)
df_RASS_metadata['Dataset'] = 'RASS'
df_RASS_metadata = df_RASS_metadata.rename(columns={'Filename': 'MorgothOutputFilename',
                                                    'RASS_value': 'RawScore'})
df_RASS_metadata['TransformedScore'] = -df_RASS_metadata['RawScore']
df_RASS_metadata = df_RASS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

## ------------------ FULL DATASET (RASS- GCS - CAMS - ICANS) -------------------------
df_all_metadata = pd.concat(
    [df_RASS_metadata, df_GCS_metadata, df_CAMS_metadata, df_ICANS_metadata], # ],
    axis=0,
    ignore_index=True
)

df_all_metadata

# **Split the dataset into train validation test using subject independent split**

In [ ]:
df_subjects = df_all_metadata[['BDSPPatientID', 'Dataset']].drop_duplicates()

from sklearn.model_selection import train_test_split

train_val, test = train_test_split(
    df_subjects,
    test_size=0.2,
    stratify=df_subjects['Dataset'],
    random_state=42
)

train, val = train_test_split(
    train_val,
    test_size=0.125,
    stratify=train_val['Dataset'],
    random_state=42
)

train_ids = set(train['BDSPPatientID'])
val_ids = set(val['BDSPPatientID'])
test_ids = set(test['BDSPPatientID'])

def assign_split(pid):
    if pid in train_ids:
        return 'Train'
    elif pid in val_ids:
        return 'Val'
    elif pid in test_ids:
        return 'Test'
    else:
        return 'None'

df_all_metadata['Split'] = df_all_metadata['BDSPPatientID'].apply(assign_split)

df_train = df_all_metadata[df_all_metadata['Split'] == 'Train'].reset_index(drop=True)
df_val   = df_all_metadata[df_all_metadata['Split'] == 'Val'].reset_index(drop=True)
df_test  = df_all_metadata[df_all_metadata['Split'] == 'Test'].reset_index(drop=True)

print("Train shape:", df_train.shape)
print("Val shape:", df_val.shape)
print("Test shape:", df_test.shape)

# **Feaure Engineering**

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm


def morgoth_output_file_location(data_group):
    base_paths = {
        "RASS": "/home/ayush/Desktop/MGB_RASS_EEGs/RASS_ALL_EEG_10min/MORGOTH_embeddings",
        "GCS": "/home/ayush/Desktop/MGB_GCS_EEGs/MORGOTH_GCS_Embeddings",
        "CAMS": "/home/ayush/Desktop/CAM-S_dataset/MORGOTH_Embeddings",
        "ICANS": "/home/ayush/Desktop/ICANS_dataset/Morgoth_embeddings_for_model"
    }

    root = base_paths[data_group]

    return {
        "SLOWING": os.path.join(root, "SLOWING"),
        "FOCGEN": os.path.join(root, "FOCGEN"),
        "IIIC": os.path.join(root, "IIIC"),
        "NM": os.path.join(root, "NM"),
        "BS": os.path.join(root, "BS"),
        "SLEEP": os.path.join(root, "SLEEP")
    }


def load_feature(path, filename):
    df = pd.read_csv(os.path.join(path, filename))
    if "pred_class" in df.columns:
        df = df.drop(columns=["pred_class"])
    return df.values


def morgoth_10minfea_matrix(data_frame):
    features = []
    file_names = []

    for row in tqdm(data_frame.itertuples(index=False), total=len(data_frame)):
        fname = row.MorgothOutputFilename
        dataset = row.Dataset

        paths = morgoth_output_file_location(dataset)

        sleep = load_feature(paths["SLEEP"], fname)
        nm = load_feature(paths["NM"], fname)
        bs = load_feature(paths["BS"], fname)

        focgen = load_feature(paths["FOCGEN"], fname)
        slowing = load_feature(paths["SLOWING"], fname)
        iiic = load_feature(paths["IIIC"], fname)

        subject_feature = np.concatenate(
            [sleep, nm, bs, focgen, slowing, iiic],
            axis=1
        )

        features.append(subject_feature)
        file_names.append(fname)

    X = np.stack(features, axis=0)

    Y_grouped = data_frame["GroupedScore"].to_numpy()
    Y_raw = data_frame["RawScore"].to_numpy()
    Y_raw_transformed = data_frame['TransformedScore'].to_numpy()

    dataset_names = data_frame["Dataset"].to_numpy()
    file_names = np.array(file_names)

    print("Grouped Label shape:", Y_grouped.shape)
    print("Raw Label shape:", Y_raw.shape)
    print("Raw Label transformed shape:", Y_raw_transformed.shape)
    print("Feature matrix shape:", X.shape)
    
    return X, Y_grouped, Y_raw, Y_raw_transformed, dataset_names, file_names

In [ ]:
X_train_data, Ygrouped_train, Yraw_train, Yraw_transformed_train, train_dataset_names,trn_filenames = morgoth_10minfea_matrix(df_train)
X_val_data, Ygrouped_val, Yraw_val, Yraw_transformed_val, val_dataset_names, val_filenames = morgoth_10minfea_matrix(df_val)
X_tst_data, Ygrouped_tst, Yraw_tst, Yraw_transformed_tst, tst_dataset_names, tst_filenames = morgoth_10minfea_matrix(df_test)

In [ ]:
X_train_data_trch = torch.tensor(X_train_data, dtype=torch.float32)
X_val_data_trch   = torch.tensor(X_val_data, dtype=torch.float32)
X_tst_data_trch   = torch.tensor(X_tst_data, dtype=torch.float32)

Ygrouped_train_trch = torch.tensor(Ygrouped_train, dtype=torch.long)
Ygrouped_val_trch   = torch.tensor(Ygrouped_val, dtype=torch.long)
Ygrouped_tst_trch   = torch.tensor(Ygrouped_tst, dtype=torch.long)

Yraw_train_trch = torch.tensor(Yraw_train, dtype=torch.long)
Yraw_val_trch   = torch.tensor(Yraw_val, dtype=torch.long)
Yraw_tst_trch   = torch.tensor(Yraw_tst, dtype=torch.long)

train_dataset_names = np.array(train_dataset_names)
val_dataset_names   = np.array(val_dataset_names)
tst_dataset_names   = np.array(tst_dataset_names)

# **Get the Embeddings from the Frozen Triplet Model**

In [ ]:
#-------------------------- ResNet-GAP only model -------------------------------------
import torch.nn.functional as F
class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(out_ch)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_ch)

        # If channels differ → use 1x1 conv for skip
        self.shortcut = nn.Sequential()
        if in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = out + identity
        return F.relu(out)

class MORGOTH_ResNet1D_onlyGAP(nn.Module):
    def __init__(self, num_features, filters=None, use_logit=True):
        super().__init__()
        self.use_logit = use_logit
        

        if filters is None:
            filters = [64, 128, 128, 256, 256]

        # Initial conv
        self.conv0 = nn.Conv1d(num_features, filters[0], kernel_size=7, padding=3)
        self.bn0 = nn.BatchNorm1d(filters[0])
        self.pool0 = nn.MaxPool1d(kernel_size=2)

        # ResNet blocks
        blocks = []
        in_ch = filters[0]
        for out_ch in filters:
            blocks.append(ResidualBlock1D(in_ch, out_ch))
            blocks.append(nn.MaxPool1d(kernel_size=2))
            in_ch = out_ch
        self.resnet_layers = nn.Sequential(*blocks)

        # GAP but DON'T squeeze yet
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(256, 40)
    def forward(self, x):
        if self.use_logit:
            eps = 1e-6
            x = torch.log((x + eps) / (1 - x + eps))

        x = x.permute(0, 2, 1)

        x = self.pool0(F.relu(self.bn0(self.conv0(x))))
        x = self.resnet_layers(x)

        # GAP: (B,C,1)
        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)
        x = F.relu(x)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# --------- LOAD MODEL ----------
model_trained = MORGOTH_ResNet1D_onlyGAP(num_features=17)
model_trained.load_state_dict(
    torch.load("/home/ayush/Desktop/NESI_review_material/24ModelBaseline/ModelCheckpoints/TripletModels/Triplet_M12_BestModel.pth",
               map_location=device,
               weights_only=True)
)
model_trained = model_trained.to(device)
model_trained.eval()

# --------- CREATE LOADERS ----------
train_ds_all = TensorDataset(X_train_data_trch)
val_ds_all   = TensorDataset(X_val_data_trch)
tst_ds_all   = TensorDataset(X_tst_data_trch)

train_dl_all = DataLoader(train_ds_all, batch_size=128, shuffle=False)
val_dl_all   = DataLoader(val_ds_all, batch_size=128, shuffle=False)
tst_dl_all   = DataLoader(tst_ds_all, batch_size=128, shuffle=False)

# --------- FUNCTION TO EXTRACT EMBEDDINGS ----------
def get_embeddings(model_trained, dataloader):
    all_embeddings = []

    with torch.no_grad():
        for (x,) in dataloader:
            x = x.to(device)
            emb = model_trained(x)
            all_embeddings.append(emb.cpu())

    return torch.cat(all_embeddings, dim=0)

# --------- GET EMBEDDINGS ----------
train_embeddings = get_embeddings(model_trained, train_dl_all)
val_embeddings   = get_embeddings(model_trained, val_dl_all)
tst_embeddings   = get_embeddings(model_trained, tst_dl_all)

print("Train embeddings shape:", train_embeddings.shape)
print("Val embeddings shape:", val_embeddings.shape)
print("Test embeddings shape:", tst_embeddings.shape)

# **Pair-wise Global Score Preduction Model Training**

## **Pairwise data sampling**

In [ ]:
import numpy as np
import random
from tqdm import tqdm


def create_pairs(
    embeddings,
    y_raw_transformed,
    y_grouped,
    dataset_names,
    mode="within_plus_global",
    k_pairs_per_anchor=4,
    hard_ratio=0.45,
    medium_ratio=0.30,
    easy_ratio=0.25,
    dataset_weights=None,
):

    N = len(embeddings)

    embeddings = np.array(embeddings)
    y_raw_transformed = np.array(y_raw_transformed)
    y_grouped = np.array(y_grouped)
    dataset_names = np.array(dataset_names)

    if dataset_weights is None:
        dataset_weights = {
            "RASS": 0.3,
            "GCS": 0.3,
            "CAMS": 0.2,
            "ICANS": 0.2,
        }

    def sample_idx(cands):
        return random.choice(cands) if len(cands) > 0 else None

    # group indices
    dataset_to_indices = {}
    for i in range(N):
        d = dataset_names[i]
        dataset_to_indices.setdefault(d, []).append(i)

    pairs = []

    total_anchors = N // 10

    dataset_anchor_counts = {
        d: int(total_anchors * dataset_weights.get(d, 0.25))
        for d in dataset_to_indices.keys()
    }

    # =========================
    # DATASET LOOP (ONLY BAR)
    # =========================
    for dataset, indices in tqdm(dataset_to_indices.items(), desc="Creating Pairs"):

        anchor_count = dataset_anchor_counts.get(dataset, len(indices))
        anchor_indices = random.sample(indices, min(anchor_count, len(indices)))

        # NO inner tqdm here → prevents spam
        for i in anchor_indices:

            emb_i = embeddings[i]

            k = k_pairs_per_anchor

            n_hard = max(1, int(k * hard_ratio))
            n_med = max(1, int(k * medium_ratio))
            n_easy = max(1, k - n_hard - n_med)

            same_dataset_idx = indices
            yi = y_raw_transformed[i]

            # ---------------- HARD ----------------
            for _ in range(n_hard):
                candidates = [
                    j for j in same_dataset_idx
                    if abs(y_raw_transformed[j] - yi) <= 1e-3
                ]

                j = sample_idx(candidates)
                if j is None or j == i:
                    continue

                y = 1 if y_raw_transformed[i] > y_raw_transformed[j] else -1

                pairs.append((emb_i, embeddings[j], y, {
                    "type": "hard_within",
                    "dataset": dataset
                }))

            # ---------------- MEDIUM ----------------
            for _ in range(n_med):
                candidates = [
                    j for j in same_dataset_idx
                    if 0.5 < abs(y_raw_transformed[j] - yi) <= 3
                ]

                j = sample_idx(candidates)
                if j is None or j == i:
                    continue

                y = 1 if y_raw_transformed[i] > y_raw_transformed[j] else -1

                pairs.append((emb_i, embeddings[j], y, {
                    "type": "medium_within",
                    "dataset": dataset
                }))

            # ---------------- EASY ----------------
            for _ in range(n_easy):
                candidates = [
                    j for j in same_dataset_idx
                    if abs(y_raw_transformed[j] - yi) > 3
                ]

                j = sample_idx(candidates)
                if j is None or j == i:
                    continue

                y = 1 if y_raw_transformed[i] > y_raw_transformed[j] else -1

                pairs.append((emb_i, embeddings[j], y, {
                    "type": "easy_within",
                    "dataset": dataset
                }))

            # ---------------- CROSS ----------------
            if mode == "within_plus_global":

                cross_candidates = [
                    j for j in range(N)
                    if dataset_names[j] != dataset
                    and y_grouped[j] != y_grouped[i]
                ]

                for _ in range(max(1, k // 2)):

                    j = sample_idx(cross_candidates)
                    if j is None:
                        continue

                    y = 1 if y_grouped[i] > y_grouped[j] else -1

                    pairs.append((emb_i, embeddings[j], y, {
                        "type": "cross_global",
                        "dataset_i": dataset,
                        "dataset_j": dataset_names[j]
                    }))

    return pairs

In [ ]:
import torch
from torch.utils.data import Dataset


class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        emb_a, emb_b, y, meta = self.pairs[idx]

        emb_a = torch.tensor(emb_a, dtype=torch.float32)
        emb_b = torch.tensor(emb_b, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        return emb_a, emb_b, y

## **Model building**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class EEGScoringModel(nn.Module):
    def __init__(self, input_dim=40, hidden_dim=30, dropout=0.3):
        super(EEGScoringModel, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)  # scalar badness score
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

## **Training code for the global score prediction model**

In [ ]:
def pairwise_loss(score_a, score_b, y):
    """
    y = +1 if A worse than B else -1
    """
    return -torch.log(torch.sigmoid(y * (score_a - score_b)) + 1e-8).mean()

In [ ]:
import torch


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=100,
    save_path="./best_model.pth",
    patience=20
):

    best_val_loss = float("inf")
    patience_counter = 0

    model.to(device)

    for epoch in range(epochs):

        # =========================
        # TRAIN
        # =========================
        model.train()
        train_loss_total = 0.0

        for emb_a, emb_b, y in train_loader:

            emb_a = emb_a.to(device)
            emb_b = emb_b.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            score_a = model(emb_a)
            score_b = model(emb_b)

            loss = pairwise_loss(score_a, score_b, y)

            loss.backward()
            optimizer.step()

            train_loss_total += loss.item()

        train_loss = train_loss_total / len(train_loader)

        # =========================
        # VALIDATION
        # =========================
        model.eval()
        val_loss_total = 0.0

        with torch.no_grad():
            for emb_a, emb_b, y in val_loader:

                emb_a = emb_a.to(device)
                emb_b = emb_b.to(device)
                y = y.to(device)

                score_a = model(emb_a)
                score_b = model(emb_b)

                loss = pairwise_loss(score_a, score_b, y)

                val_loss_total += loss.item()

        val_loss = val_loss_total / len(val_loader)

        # =========================
        # LOGGING
        # =========================
        print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # =========================
        # CHECKPOINT
        # =========================
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss
            }, save_path)

            print("✔ Saved best model")

        else:
            patience_counter += 1

        # =========================
        # EARLY STOPPING
        # =========================
        if patience_counter >= patience:
            print("⛔ Early stopping triggered")
            break

# **Implement the Global score prediction model**

## **Dataset prearation for pairwise training and dataloader creation**

In [ ]:
train_pairs = create_pairs(
    embeddings=train_embeddings,
    y_raw_transformed=Yraw_transformed_train,
    y_grouped=Ygrouped_train,
    dataset_names=train_dataset_names,
    mode="within_plus_global"
)

val_pairs = create_pairs(
    embeddings=val_embeddings,
    y_raw_transformed=Yraw_transformed_val,
    y_grouped=Ygrouped_val,
    dataset_names=val_dataset_names,
    mode="within_plus_global"
)

train_dataset = PairDataset(train_pairs)
val_dataset = PairDataset(val_pairs)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)


## **Train the model**

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"

model = EEGScoringModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=100,
    save_path="/home/ayush/Desktop/NESI_review_material/24ModelBaseline/ModelCheckpoints/NESIScoreModels/NESIScore_M12.pth",
    patience=20
)